# Lab 1 — DVM Color Classification

Классификация цвета автомобиля по фронтальному фото (датасет DVM, top-6 цветов: `black, grey, white, blue, silver, red`).

**Цель:** обучить и сравнить 3 модели (1 from-scratch + 2 pretrained), получить F1_macro > 0.8.

**Запуск:** `Runtime → Change runtime type → GPU`, затем выполни ячейки по порядку. Все ячейки идемпотентны — переисполнение не ломает состояние.

## 1. Clone / Pull репозитория

В Colab нужен secret `GITHUB_TOKEN` (Tools → Secrets).

In [ ]:
import os
from google.colab import userdata

TOKEN = userdata.get('GITHUB_TOKEN')
REPO = "Ma-XD/ITMO-CV"
REPO_DIR = "/content/ITMO-CV"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://{TOKEN}@github.com/{REPO}.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"📂 {os.getcwd()}")

## 2. Монтирование Google Drive

Делается в ячейке ноутбука (а не из `!python ...`) — `drive.mount` требует IPython-ядро.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Setup (GPU, зависимости)

In [ ]:
!python colab_setup.py

## 4. Verify окружения

In [ ]:
from env_config import print_env
print_env()

## 5. Загрузка датасета с Drive

Драйв медленный для случайного чтения мелких файлов (61k jpg). Поэтому архив **копируем** один раз с Drive на локальный SSD VM (`/content/`) и распаковываем туда же — обучение читает из `/content/data/dvm/...`, а на Drive ничего не пишем.

Источник: `MyDrive/ITMO-CV/lab1-CLAS/data/dvm_confirmed_fronts.zip` (765 МБ, без сжатия).

In [ ]:
%%bash
set -e
SRC=/content/drive/MyDrive/ITMO-CV/lab1-CLAS/data/dvm_confirmed_fronts.zip
DST=/content/dvm_confirmed_fronts.zip
DATA_DIR=/content/data

if [ ! -f "$SRC" ]; then
  echo "❌ Не найден архив на Drive: $SRC"
  exit 1
fi

if [ ! -f "$DST" ]; then
  echo "📥 cp с Drive на локальный SSD..."
  cp "$SRC" "$DST"
else
  echo "✅ Архив уже скопирован: $DST"
fi
ls -lh "$DST"

mkdir -p "$DATA_DIR"
if [ ! -d "$DATA_DIR/dvm/confirmed_fronts" ]; then
  echo "📦 unzip → $DATA_DIR ..."
  unzip -q "$DST" -d "$DATA_DIR"
else
  echo "✅ Уже распаковано: $DATA_DIR/dvm/confirmed_fronts"
fi

echo "---"
echo "Изображений: $(find "$DATA_DIR/dvm/confirmed_fronts" -name '*.jpg' | wc -l)  (ожидаем 61827)"

## 6. Сборка `index.csv`

Парсит filename'ы по `$$`, фильтрует `unlisted`/`multicolour` и off-target цвета, оставляет только top-6 → `/content/data/dvm/index.csv`. Идемпотентно: если CSV уже есть, скрипт его не перезаписывает (для пересборки — `--force`).

In [ ]:
!python lab1-CLAS/scripts/build_index.py

## 7. Smoke-test data pipeline

Проверяем, что `data.py` корректно строит сплиты, лоадеры собирают батчи и `WeightedRandomSampler` балансирует классы.

In [ ]:
import sys
from collections import Counter

LAB_DIR = "/content/ITMO-CV/lab1-CLAS"
if LAB_DIR not in sys.path:
    sys.path.insert(0, LAB_DIR)

from config import TARGET_COLORS
from data import get_dataloaders, load_index, make_splits

df = load_index()
print(f"index: {len(df)} rows, classes: {TARGET_COLORS}")

splits = make_splits(df)
print(f"splits — train: {len(splits.train)} | val: {len(splits.val)} | test: {len(splits.test)}")

loaders = get_dataloaders(splits, batch_size=32, num_workers=2)
x, y = next(iter(loaders["train"]))
print(f"batch x={tuple(x.shape)} {x.dtype}, y={tuple(y.shape)} sample={y.tolist()[:8]}")

print("\n[sampler check] доли классов за 100 train-батчей (цель ~16.7% на класс):")
counts = Counter()
for i, (_, yb) in enumerate(loaders["train"]):
    counts.update(yb.tolist())
    if i >= 99:
        break
total = sum(counts.values())
for c in TARGET_COLORS:
    idx = TARGET_COLORS.index(c)
    print(f"  [{idx}] {c:<8} {counts[idx]/total*100:5.2f}%")